# Bricks and mortar: elements are nodes, joints are edges

Companion to [Week 06 §8](../README.md#8-graphs-of-building-elements).
Until now a node was a room or a point. Here a node is a **thing you can
pick up**: a brick. An edge is what holds two of them together: a mortar
joint, with its own geometry. Two bricks first, then a small pier.

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in Lecture/notebooks/ or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_plan, draw_points, ramp, row, view3d

import compas.geometry as cg
from compas.colors import Color
from compas.datastructures import Graph, Mesh

BRICK = [0.230, 0.110, 0.050]                 # length, width, height in metres
MORTAR = 0.010
LAYER = BRICK[2] + MORTAR
REACH = (BRICK[0] ** 2 + BRICK[1] ** 2 + BRICK[2] ** 2) ** 0.5 - 0.01      # farther apart than this: no joint

## Step 1 — two bricks
A brick is a box on a **frame** (Week 04): the frame says where it is and
which way it is turned. The second brick sits one layer up, turned by 45°.

In [ ]:
def brick_on(frame):
    return cg.Box(BRICK[0], BRICK[1], BRICK[2], frame)


def footprint(box):
    """The four corners of a brick, seen from above."""
    half_l, half_w = box.xsize / 2, box.ysize / 2
    return [box.frame.to_world_coordinates([sx * half_l, sy * half_w, 0])[:2]
            for sx, sy in ((-1, -1), (1, -1), (1, 1), (-1, 1))]


brick_1 = brick_on(cg.Frame([0, 0, 0], [1, 0, 0], [0, 1, 0]))
brick_2 = brick_on(cg.Frame([0, 0.01, LAYER], [1, 1, 0], [-1, 1, 0]))

plan = draw_points([p for b in (brick_1, brick_2) for p in footprint(b)], "two bricks, from above",
                   polygons=[footprint(brick_1), footprint(brick_2)],
                   polygon_fills=["#ffd8a8", "#a5d8ff"], size=(460, 340))
plan

In [ ]:
view3d((brick_1, {"color": Color.from_hex("#ffa94d")}), (brick_2, {"color": Color.from_hex("#74c0fc")}))

## Step 2 — the mortar between them
The joint is the space where the two bricks face each other. A trick finds
it: make both bricks **taller by the mortar thickness**, and intersect
them. What the swollen bricks share is the mortar.

The intersection is a mesh boolean, which is slow, so pairs that are too
far apart to touch are skipped before it is tried.

In [ ]:
def mortar_between(a, b):
    """The mortar joint between two bricks as a Mesh, or None if they do not face each other."""
    if a.frame.point.distance_to_point(b.frame.point) > REACH:
        return None
    swollen = [cg.Box(BRICK[0], BRICK[1], BRICK[2] + 2 * MORTAR, box.frame) for box in (a, b)]
    shared = cg.boolean_intersection_mesh_mesh(*[box.to_mesh(True, 10, 10).to_vertices_and_faces() for box in swollen])
    joint = Mesh.from_vertices_and_faces(*shared)
    return joint if joint.number_of_faces() else None


def outline(mesh):
    """The outline of a mesh seen from above."""
    return [p[:2] for p in cg.convex_hull_xy([mesh.vertex_coordinates(v) for v in mesh.vertices()])]


joint = mortar_between(brick_1, brick_2)
print(f"mortar joint: {joint.number_of_vertices()} vertices, {joint.volume() * 1e6:.0f} cm3")
draw_points(outline(joint), "the joint (grey) is where the two footprints overlap",
            polygons=[footprint(brick_1), footprint(brick_2), outline(joint)],
            polygon_fills=["#ffd8a8", "#a5d8ff", "#868e96"], size=(460, 340))

In [ ]:
view3d((brick_1, {"color": Color.from_hex("#ffa94d")}), (brick_2, {"color": Color.from_hex("#74c0fc")}),
       (joint, {"facecolor": Color.from_hex("#868e96")}))

## Step 3 — put it in a graph
Three objects, and a relation between them that so far lives only in our
heads. A graph makes the relation data:

- a **node** per brick, holding the brick itself as an attribute
- an **edge** per joint, holding the mortar

The mortar belongs on the edge because it belongs to the *pair*: remove
either brick and that joint is gone.

In [ ]:
stack = Graph()
for name, brick, layer in (("lower", brick_1, 0), ("upper", brick_2, 1)):
    stack.add_node(name, x=brick.frame.point.x, y=brick.frame.point.y, z=brick.frame.point.z,
                   brick=brick, layer=layer)
stack.add_edge("upper", "lower", mortar=joint, volume=float(joint.volume()))

print("nodes:", list(stack.nodes()), "| edges:", list(stack.edges()))
print("the upper brick touches:", stack.neighbors("upper"))
print("on that edge:", {k: (v if k == "volume" else type(v).__name__)
                        for k, v in stack.edge_attributes(("upper", "lower")).items()})

picture = stack.copy()
picture.node_attributes("lower", "xy", [0, 0])
picture.node_attributes("upper", "xy", [0, 1])
row(plan, draw_graph(picture, "the same two bricks as a graph", size=(300, 340)))

## Step 4 — a pier: the graph finds itself
Six bricks, two per layer, each layer turned 90° to the one below. Nobody
types the edges this time. Every pair of bricks is offered to
`mortar_between`, and **a joint exists where it returns something.**

In [ ]:
pier = Graph()
for layer in range(3):
    for side in (-1, 1):
        offset = side * (BRICK[1] + MORTAR) / 2
        if layer % 2 == 0:
            frame = cg.Frame([0, offset, layer * LAYER], [1, 0, 0], [0, 1, 0])
        else:
            frame = cg.Frame([offset, 0, layer * LAYER], [0, 1, 0], [-1, 0, 0])
        pier.add_node(len(list(pier.nodes())), x=frame.point.x, y=frame.point.y, z=frame.point.z,
                      brick=brick_on(frame), layer=layer)

pairs = [(a, b) for a in pier.nodes() for b in pier.nodes() if a < b]
for a, b in pairs:
    joint = mortar_between(pier.node_attribute(a, "brick"), pier.node_attribute(b, "brick"))
    if joint:
        pier.add_edge(b, a, mortar=joint, volume=float(joint.volume()))

print(f"{pier.number_of_nodes()} bricks, {len(pairs)} pairs tried, {pier.number_of_edges()} joints found")
shade = ["#ffd8a8", "#a5d8ff", "#b2f2bb"]            # one light colour per layer, for the plans
strong = ["#e8590c", "#1c7ed6", "#2f9e44"]           # and its strong version, for nodes and 3D
row(*[draw_points([], f"layer {layer}", size=(250, 250), bounds=[(-0.13, -0.13), (0.13, 0.13)],
                  polygons=[footprint(pier.node_attribute(n, "brick")) for n in pier.nodes_where(layer=layer)],
                  polygon_fills=[shade[layer]] * 2) for layer in range(3)])

### The pier as a graph
Drawn by layer, bottom to top. Bricks in the same layer are **not**
joined: the swelling trick only looks up and down, so it finds bed joints
and misses the head joint between neighbours. Knowing what your edge
detector cannot see is part of using it.

In [ ]:
picture = pier.copy()
for n in pier.nodes():
    picture.node_attributes(n, "xy", [n % 2, pier.node_attribute(n, "layer")])
colours = {n: strong[pier.node_attribute(n, "layer")] for n in pier.nodes()}
draw_graph(picture, "6 bricks, 8 bed joints", node_colors=colours, size=(380, 360))

In [ ]:
view3d(*[(pier.node_attribute(n, "brick"), {"color": Color.from_hex(colours[n])}) for n in pier.nodes()],
       *[(pier.edge_attribute(e, "mortar"), {"facecolor": Color.from_hex("#868e96")}) for e in pier.edges()])

## Step 5 — ask the graph
With the pier in a graph, quantities and checks are loops over nodes and
edges, and none of them touches the geometry again.

In [ ]:
total = sum(pier.edges_attribute("volume"))
print(f"mortar in bed joints: {total * 1e6:.0f} cm3 in {pier.number_of_edges()} joints")
for n in pier.nodes():
    print(f"brick {n} (layer {pier.node_attribute(n, 'layer')}): {pier.degree(n)} joints, "
          f"rests on {sorted(pier.neighbors_out(n))}, carries {sorted(pier.neighbors_in(n))}")
loose = [n for n in pier.nodes() if pier.degree(n) == 0]
print("bricks joined to nothing:", loose)

Each edge was added **from the upper brick to the lower one**, so
`neighbors_out` reads "rests on" and `neighbors_in` reads "carries". The
next notebook, `05_brick_wall_assembly`, builds a whole wall on that one
convention.

## Your turn
1. Turn the upper brick of Step 1 by 90° instead of 45°. Predict the mortar
   volume before you run it.
2. Add a fourth layer to the pier. How many pairs are tried now, and how
   many joints are found? Which number grows faster?
3. Slide one brick of layer 1 outwards by 0.2 m. What does the last cell
   report?
4. Find the head joints: swell the bricks sideways instead of upwards.
   Store them as edges with `kind="head"`.